# Pre-development Landcover
For each NHDA and reference area, the code determines the dominant CORINE land cover within the area in 2012 (PRE-DEVELOPMENT) and within the 100-meter surrounding area in 2021 (SURROUNDING LANDCOVER), including the proportion of the area and the class designation. It then applies the respective NHDA and RA values to both geometries, compares the land cover of the pairs, and exports the results as a GeoPackage and CSV file.

In [ ]:
import geopandas as gpd
import pandas as pd
from pathlib import Path
from datetime import datetime
from shapely.strtree import STRtree

INPUT_FILE  = r"C:\Users\agz90fk\Documents\EO4CAM\3_Daten\Output\Masterarbeit\Comparison_NHDA_RA_v2\Comparison_LST_NDVI_DEGURB.gpkg"
CORINE_2012 = r"C:\Users\agz90fk\Documents\EO4CAM\3_Daten\processed_datasets\CORINE_Bavaria\CORINE_Bavaria_2012.gpkg"
LEGEND_FILE = r"C:\Users\agz90fk\Documents\EO4CAM\3_Daten\Input\CORINE_Land_Cover_DE\CLC_legend.csv"

print("Loading data...")
clusters = gpd.read_file(INPUT_FILE)
corine   = gpd.read_file(CORINE_2012)
print(f"  → {len(clusters)} rows, {clusters['nhda_id'].nunique()} unique nhda_ids")
print(f"  → type values: {clusters['type'].unique()}")

if corine.crs != clusters.crs:
    corine = corine.to_crs(clusters.crs)

# ── Build spatial index ────────────────────────────────────────────────────
tree = STRtree(corine.geometry)

legend = pd.read_csv(LEGEND_FILE, sep=';')
legend['CLC_CODE'] = legend['CLC_CODE'].astype(str)
label_map = dict(zip(legend['CLC_CODE'], legend['LABEL3']))

AREA_TOL = 1e-6
clc_2012_results = []
start = datetime.now()
total = len(clusters)

# ── Step 1: Compute CLC 2012 for every row ─────────────────────────────────
print(f"\nStep 1: Computing CLC 2012 for all {total} rows...")
for i, (idx, row) in enumerate(clusters.iterrows()):
    if i % 100 == 0:
        print(f"  {i+1}/{total}...")

    geom = row.geometry
    candidate_idx = tree.query(geom, predicate='intersects')

    if len(candidate_idx) == 0:
        clc_2012_results.append({'clc_2012': None, 'clc_2012_pct': None})
        continue

    cluster_area = geom.area
    class_areas = {}
    for ci in candidate_idx:
        corine_row = corine.iloc[ci]
        clc_code = str(corine_row["CLC_CODE"])
        area = geom.intersection(corine_row.geometry).area
        if area > AREA_TOL:
            class_areas[clc_code] = class_areas.get(clc_code, 0.0) + area

    if not class_areas:
        clc_2012_results.append({'clc_2012': None, 'clc_2012_pct': None})
        continue

    dominant = max(class_areas, key=class_areas.get)
    pct = round(class_areas[dominant] / cluster_area * 100, 2)
    clc_2012_results.append({'clc_2012': dominant, 'clc_2012_pct': pct})

result_df = pd.DataFrame(clc_2012_results, index=clusters.index)
clusters['clc_2012']       = result_df['clc_2012']
clusters['clc_2012_pct']   = result_df['clc_2012_pct']
clusters['clc_2012_label'] = clusters['clc_2012'].map(label_map)
print(f"  ✓ Filled: {clusters['clc_2012'].notna().sum()} / {total}")

# ── Step 2: Cross-join NHDA ↔ RA values by nhda_id ────────────────────────
print("\nStep 2: Adding partner CLC values...")

nhda_lookup = (
    clusters[clusters['type'] == 'NHDA'][['nhda_id', 'clc_2012', 'clc_2012_pct', 'clc_2012_label']]
    .rename(columns={'clc_2012': 'nhda_clc_2012', 'clc_2012_pct': 'nhda_clc_2012_pct', 'clc_2012_label': 'nhda_clc_2012_label'})
)
ra_lookup = (
    clusters[clusters['type'] == 'RA'][['nhda_id', 'clc_2012', 'clc_2012_pct', 'clc_2012_label']]
    .rename(columns={'clc_2012': 'ra_clc_2012', 'clc_2012_pct': 'ra_clc_2012_pct', 'clc_2012_label': 'ra_clc_2012_label'})
)

clusters = clusters.merge(nhda_lookup, on='nhda_id', how='left')
clusters = clusters.merge(ra_lookup,   on='nhda_id', how='left')
clusters = clusters.drop(columns=['clc_2012', 'clc_2012_pct', 'clc_2012_label'])
clusters = gpd.GeoDataFrame(clusters, crs=gpd.read_file(INPUT_FILE).crs)

# ── Summary ────────────────────────────────────────────────────────────────
print(f"\nDone in {datetime.now() - start}")
print(f"\nNHDA rows — CLC 2012 distribution:")
print(clusters[clusters['type']=='NHDA']['nhda_clc_2012_label'].value_counts().head(5).to_string())
print(f"\nRA rows — CLC 2012 distribution:")
print(clusters[clusters['type']=='RA']['ra_clc_2012_label'].value_counts().head(5).to_string())
print(f"\nRows where NHDA and RA CLC 2012 differ: "
      f"{(clusters['nhda_clc_2012'] != clusters['ra_clc_2012']).sum()}")


# Surrounding Landcover

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
from shapely.strtree import STRtree
from shapely.geometry import box

OUTPUT_FILE = r"C:\Users\agz90fk\Documents\EO4CAM\3_Daten\Output\Masterarbeit\Comparison_NHDA_RA_v2\Comparison_LST_NDVI_DEGURB_CLC.gpkg"
CORINE_2021 = r"C:\Users\agz90fk\Documents\EO4CAM\3_Daten\processed_datasets\CORINE_Bavaria\CORINE_Bavaria_2021.gpkg"
LEGEND_FILE = r"C:\Users\agz90fk\Documents\EO4CAM\3_Daten\Input\CORINE_Land_Cover_DE\CLC_legend.csv"

BUFFER_DIST = 100
TILE_SIZE   = 20_000   # 20km grid tiles

# ── Load ────────────────────────────────────────────────────────────────────
print("Loading CORINE 2021...")
t0 = datetime.now()
corine = gpd.read_file(CORINE_2021)
if corine.crs != clusters.crs:
    corine = corine.to_crs(clusters.crs)
print(f"  → {len(corine)} features ({datetime.now() - t0})")

legend = pd.read_csv(LEGEND_FILE, sep=';')
legend['CLC_CODE'] = legend['CLC_CODE'].astype(str)
label_map = dict(zip(legend['CLC_CODE'], legend['LABEL3']))

# ── Tile CORINE and build STRtree ───────────────────────────────────────────
print(f"\nTiling CORINE into {TILE_SIZE/1000:.0f}km grid...")
t0 = datetime.now()
minx, miny, maxx, maxy = clusters.total_bounds
xs = np.arange(minx - BUFFER_DIST, maxx + TILE_SIZE, TILE_SIZE)
ys = np.arange(miny - BUFFER_DIST, maxy + TILE_SIZE, TILE_SIZE)
tiles = gpd.GeoDataFrame(
    geometry=[box(x, y, x + TILE_SIZE, y + TILE_SIZE) for x in xs for y in ys],
    crs=clusters.crs
)
tiled = gpd.overlay(corine[['CLC_CODE', 'geometry']], tiles, how='intersection', keep_geom_type=False)
tiled = tiled[tiled.geometry.area > 1].reset_index(drop=True)
print(f"  → {len(tiled)} tiled pieces ({datetime.now() - t0})")

print("Building spatial index...")
tree = STRtree(tiled.geometry)

# ── Step 1: Compute surrounding CLC 2021 for every row ────────────────────
print(f"\nStep 1: Processing {len(clusters)} rows...")
AREA_TOL = 1e-6
start = datetime.now()
total = len(clusters)
sur_results = []

for i, (idx, row) in enumerate(clusters.iterrows()):
    geom = row.geometry
    ring_geom = geom.buffer(BUFFER_DIST).difference(geom)
    ring_area = ring_geom.area

    if ring_area <= AREA_TOL:
        sur_results.append({'sur_clc_2021': None, 'sur_clc_2021_pct': None})
        continue

    candidate_idx = tree.query(ring_geom, predicate='intersects')
    if len(candidate_idx) == 0:
        sur_results.append({'sur_clc_2021': None, 'sur_clc_2021_pct': None})
        continue

    class_areas = {}
    for ci in candidate_idx:
        t_row = tiled.iloc[ci]
        clc_code = str(t_row["CLC_CODE"])
        area = ring_geom.intersection(t_row.geometry).area
        if area > AREA_TOL:
            class_areas[clc_code] = class_areas.get(clc_code, 0.0) + area

    if not class_areas:
        sur_results.append({'sur_clc_2021': None, 'sur_clc_2021_pct': None})
        continue

    dominant = max(class_areas, key=class_areas.get)
    pct = round(class_areas[dominant] / ring_area * 100, 2)
    sur_results.append({'sur_clc_2021': dominant, 'sur_clc_2021_pct': pct})

    if (i + 1) % 100 == 0:
        elapsed = (datetime.now() - start).total_seconds()
        eta = (total - i - 1) / ((i + 1) / elapsed)
        print(f"  [{i+1:>4}/{total}]  elapsed: {elapsed:.0f}s  ETA: {eta:.0f}s")

sur_df = pd.DataFrame(sur_results, index=clusters.index)
clusters['sur_clc_2021']       = sur_df['sur_clc_2021']
clusters['sur_clc_2021_pct']   = sur_df['sur_clc_2021_pct']
clusters['sur_clc_2021_label'] = clusters['sur_clc_2021'].map(label_map)
print(f"  ✓ Filled: {clusters['sur_clc_2021'].notna().sum()} / {total}")

# ── Step 2: Cross-join NHDA ↔ RA values by nhda_id ────────────────────────
print("\nStep 2: Adding partner CLC values...")

nhda_lookup = (
    clusters[clusters['type'] == 'NHDA'][['nhda_id', 'sur_clc_2021', 'sur_clc_2021_pct', 'sur_clc_2021_label']]
    .rename(columns={'sur_clc_2021': 'nhda_sur_clc_2021', 'sur_clc_2021_pct': 'nhda_sur_clc_2021_pct', 'sur_clc_2021_label': 'nhda_sur_clc_2021_label'})
)
ra_lookup = (
    clusters[clusters['type'] == 'RA'][['nhda_id', 'sur_clc_2021', 'sur_clc_2021_pct', 'sur_clc_2021_label']]
    .rename(columns={'sur_clc_2021': 'ra_sur_clc_2021', 'sur_clc_2021_pct': 'ra_sur_clc_2021_pct', 'sur_clc_2021_label': 'ra_sur_clc_2021_label'})
)

clusters = clusters.merge(nhda_lookup, on='nhda_id', how='left')
clusters = clusters.merge(ra_lookup,   on='nhda_id', how='left')
clusters = clusters.drop(columns=['sur_clc_2021', 'sur_clc_2021_pct', 'sur_clc_2021_label'])
clusters = gpd.GeoDataFrame(clusters, crs=clusters.crs)

# ── Summary ────────────────────────────────────────────────────────────────
print(f"\nDone in {datetime.now() - start}")
print(f"\nNHDA rows — surrounding CLC 2021 distribution:")
print(clusters[clusters['type']=='NHDA']['nhda_sur_clc_2021_label'].value_counts().head(5).to_string())
print(f"\nRA rows — surrounding CLC 2021 distribution:")
print(clusters[clusters['type']=='RA']['ra_sur_clc_2021_label'].value_counts().head(5).to_string())
print(f"\nRows where NHDA and RA surrounding CLC 2021 differ: "
      f"{(clusters['nhda_sur_clc_2021'] != clusters['ra_sur_clc_2021']).sum()}")

# ── Save ─────────────────────────────────────────────────────────────────────
output_path = Path(OUTPUT_FILE)
output_path.parent.mkdir(parents=True, exist_ok=True)
clusters.to_file(OUTPUT_FILE, driver="GPKG")
clusters.drop(columns="geometry").to_csv(output_path.with_suffix(".csv"), index=False)
print(f"\n✓ Saved: {OUTPUT_FILE}")
